# Model B: 5-fold cross-validation of DeBERTa

Train and validation pairs are combined (3,860 pairs, 1,000 stories) and split into 5 folds **by story**. DeBERTa is trained 5 times, each time predicting the fold it did not see, so every pair gets an out-of-fold prediction. Settings are fixed in advance (learning rate 1e-5, 3 epochs, seed 42) and the **last** epoch is always used, so nothing is tuned on the scored data.

Each fold is saved as soon as it finishes. If the session dies, run all cells again: finished folds are skipped.

About 35 minutes. **Kaggle:** Accelerator GPU T4, Internet On, then Save Version and Save & Run All (Commit). **Colab:** Runtime, then T4 GPU.

## 1. Check the GPU

In [1]:
!nvidia-smi

Thu Nov 19 18:27:38 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.178.04             Driver Version: 580.178.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   39C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 2. Install packages

In [2]:
!pip install -q sentencepiece protobuf

## 3. Storage and data
Needs `pairs_cv.csv` (made on your laptop by `python -m src.make_folds`).

In [3]:
import glob, os, shutil

if os.path.exists("/kaggle"):
    # Kaggle: files in /kaggle/working are kept when you use "Save Version".
    OUT_DIR = "/kaggle/working"
    found = glob.glob("/kaggle/input/**/pairs_cv.csv", recursive=True)
    assert found, "Add your dataset containing pairs_cv.csv (Add Input, right panel)"
    shutil.copy(found[0], f"{OUT_DIR}/pairs_cv.csv")
else:
    # Colab: everything is saved to Google Drive, so a disconnect loses nothing.
    from google.colab import drive, files
    drive.mount("/content/drive")
    OUT_DIR = "/content/drive/MyDrive/good-news-nlp-cv"
    os.makedirs(OUT_DIR, exist_ok=True)
    if not os.path.exists(f"{OUT_DIR}/pairs_cv.csv"):
        print("First time: choose pairs_cv.csv")
        uploaded = files.upload()
        name = next(n for n in uploaded if n.startswith("pairs_cv"))
        shutil.move(name, f"{OUT_DIR}/pairs_cv.csv")

print("Saving to:", OUT_DIR)
print("Folds already finished:", sorted(os.path.basename(f) for f in glob.glob(f"{OUT_DIR}/deberta_cv_fold*.csv")))

Saving to: /kaggle/working
Folds already finished: []


## 4. Settings

In [4]:
import glob, os, random, time
import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader, Dataset
from transformers import AutoModelForSequenceClassification, AutoTokenizer, get_linear_schedule_with_warmup

BASE = {
    "model_name": "microsoft/deberta-v3-base",
    "max_length": 64,
    "batch_size": 16,
    "weight_decay": 0.01,
    "warmup_ratio": 0.1,
}
# Settings fixed in advance (chosen in the multi-seed runs): lr 1e-5, 3 epochs.
# One run per fold; the FINAL epoch is used, so nothing is picked on the scored fold.
LEARNING_RATE, EPOCHS, SEED, N_FOLDS = 1e-5, 3, 42, 5
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"


def set_seed(seed: int) -> None:
    """Fix every source of randomness so a rerun gives the same result."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


print("Device:", DEVICE, "| folds:", N_FOLDS)

Device: cuda | folds: 5


## 5. Load the folds and the tokenizer
Should show 5 folds of about 771 pairs each.

In [5]:
pairs = pd.read_csv(f"{OUT_DIR}/pairs_cv.csv")
print(f"Pairs: {len(pairs)} in {pairs['story_group'].nunique()} stories")
print(pairs.groupby("fold").size().rename("pairs").to_string())


class PairDataset(Dataset):
    """(headline_a, headline_b, label); swap=True also adds every pair reversed."""

    def __init__(self, df: pd.DataFrame, swap: bool):
        a, b, y = list(df["headline_a"]), list(df["headline_b"]), list(df["label"])
        if swap:
            a, b, y = a + b, b + a, y + [1 - v for v in y]
        self.a, self.b, self.y = a, b, y

    def __len__(self):
        return len(self.y)

    def __getitem__(self, i):
        return self.a[i], self.b[i], self.y[i]


tokenizer = AutoTokenizer.from_pretrained(BASE["model_name"])


def collate(batch):
    """Tokenise a batch as '[CLS] A [SEP] B [SEP]'."""
    a, b, y = zip(*batch)
    enc = tokenizer(list(a), list(b), truncation=True, max_length=BASE["max_length"],
                    padding=True, return_tensors="pt")
    enc["labels"] = torch.tensor(y)
    return enc

Pairs: 3860 in 1000 stories
fold
0    771
1    772
2    771
3    774
4    772


config.json:   0%|          | 0.00/579 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

spm.model: reconstructing file:   0%|          |  0.00B / 2.46MB            

spm.model: downloading bytes:           |  0.00B            

## 6. Training functions
Fresh DeBERTa in float32 for every fold; stops if the loss becomes NaN.

In [6]:
@torch.no_grad()
def predict_proba(model, df: pd.DataFrame) -> np.ndarray:
    """P(headline_a wins), averaged over both orders for symmetry."""
    model.eval()

    def run(a_col, b_col):
        ds = PairDataset(df.rename(columns={a_col: "headline_a", b_col: "headline_b"}), swap=False)
        probs = []
        for batch in DataLoader(ds, batch_size=64, collate_fn=collate):
            batch = {k: v.to(DEVICE) for k, v in batch.items() if k != "labels"}
            probs.append(torch.softmax(model(**batch).logits.float(), dim=-1)[:, 1].cpu())
        return torch.cat(probs).numpy()

    return (run("headline_a", "headline_b") + (1 - run("headline_b", "headline_a"))) / 2


def train_one(train_df: pd.DataFrame, val_df: pd.DataFrame, learning_rate: float, epochs: int, seed: int):
    """Train one DeBERTa from scratch on train_df; return (history, final-epoch probabilities on val_df)."""
    set_seed(seed)
    # float32 on purpose: transformers 5 loads DeBERTa in float16, which gives NaN loss.
    model = AutoModelForSequenceClassification.from_pretrained(
        BASE["model_name"], num_labels=2, dtype=torch.float32).to(DEVICE)
    loader = DataLoader(PairDataset(train_df, swap=True), batch_size=BASE["batch_size"],
                        shuffle=True, collate_fn=collate,
                        generator=torch.Generator().manual_seed(seed))
    optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate,
                                  weight_decay=BASE["weight_decay"])
    total = len(loader) * epochs
    scheduler = get_linear_schedule_with_warmup(optimizer, int(BASE["warmup_ratio"] * total), total)

    history, best_probs = [], None
    for epoch in range(1, epochs + 1):
        model.train()
        loss_sum = 0.0
        for batch in loader:
            batch = {k: v.to(DEVICE) for k, v in batch.items()}
            loss = model(**batch).loss
            if torch.isnan(loss):
                raise RuntimeError("NaN loss: check the model loaded in float32")
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()
            loss_sum += loss.item()
        probs = predict_proba(model, val_df)
        acc = ((probs > 0.5).astype(int) == val_df["label"].to_numpy()).mean()
        history.append({"epoch": epoch,
                        "train_loss": loss_sum / len(loader), "val_accuracy": acc})
        print(f"  epoch {epoch}: train loss {loss_sum / len(loader):.4f}, val accuracy {acc:.3f}")
        best_probs = probs  # keep the last epoch: no selection on the scored fold
    del model, optimizer
    torch.cuda.empty_cache()
    return history, best_probs

## 7. Train the 5 folds

In [7]:
start = time.time()
for k in range(N_FOLDS):
    pred_path = f"{OUT_DIR}/deberta_cv_fold{k}.csv"
    if os.path.exists(pred_path):
        print(f"Fold {k}: already saved, skipping")
        continue
    train_df = pairs[pairs["fold"] != k].reset_index(drop=True)
    val_df = pairs[pairs["fold"] == k].reset_index(drop=True)
    print(f"Fold {k}: train {len(train_df)} pairs, predict {len(val_df)} pairs")
    history, probs = train_one(train_df, val_df, LEARNING_RATE, EPOCHS, SEED)
    out = val_df[["test_id", "story_group", "fold", "headline_a", "headline_b", "label"]].copy()
    out["prob_a_wins"] = probs
    out["prediction"] = (probs > 0.5).astype(int)
    out["correct"] = out["prediction"] == out["label"]
    out.to_csv(pred_path, index=False)  # saved straight away
    pd.DataFrame(history).assign(fold=k).to_csv(f"{OUT_DIR}/deberta_cv_history_fold{k}.csv", index=False)
    print(f"  fold {k} accuracy {out['correct'].mean():.3f}  "
          f"(saved, {(time.time() - start) / 60:.1f} min this session)\n")
print("All folds done.")

Fold 0: train 3089 pairs, predict 771 pairs


pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  371MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  371MB            

model.safetensors: downloading bytes:           |  0.00B            

... (10 outputs left out)

  epoch 1: train loss 0.6948, val accuracy 0.664
  epoch 2: train loss 0.6007, val accuracy 0.756
  epoch 3: train loss 0.2677, val accuracy 0.760
  fold 3 accuracy 0.760  (saved, 20.8 min this session)

Fold 4: train 3088 pairs, predict 772 pairs


Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

[transformers] DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     | 
----------------------------------------+------------+-
lm_predictions.lm_head.bias             | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
classifier.bias                         | MISSING    | 
pooler.dense.bias                       | MISSING    | 
classifier.weight                       | MISSING    | 
pooler.den

  epoch 1: train loss 0.6954, val accuracy 0.605
  epoch 2: train loss 0.5994, val accuracy 0.732
  epoch 3: train loss 0.2919, val accuracy 0.747
  fold 4 accuracy 0.747  (saved, 26.0 min this session)

All folds done.


## 8. Combine the folds
Writes `cv_predictions_deberta.csv`: download it into `data/processed` on your laptop.

In [8]:
folds = [pd.read_csv(f) for f in sorted(glob.glob(f"{OUT_DIR}/deberta_cv_fold*.csv"))]
assert len(folds) == N_FOLDS, f"Only {len(folds)} of {N_FOLDS} folds finished: run cell 7 again"
cv = pd.concat(folds, ignore_index=True)
cv.to_csv(f"{OUT_DIR}/cv_predictions_deberta.csv", index=False)
print(f"DeBERTa cross-validated accuracy: {cv['correct'].mean():.3f}")
print(cv.groupby("fold")["correct"].mean().round(3).to_string())

if not os.path.exists("/kaggle"):
    files.download(f"{OUT_DIR}/cv_predictions_deberta.csv")

DeBERTa cross-validated accuracy: 0.752
fold
0    0.765
1    0.754
2    0.733
3    0.760
4    0.747
